In [1]:
import pandas as pd
import re
df = pd.read_csv('data/communards_full_dataset.csv')

df.head()

,Profile_URL,Raw_Bio_Text,Lieu de naissance,Date de naissance déclarée,Âge déclaré,Âge calculé au 31 mai 1871,État civil,Père,Mère,Adresse,...,Archives du Service historique de la Défense,Archives nationales d’Outre-mer,"Archives nationales, dossiers de grâce","Archives nationales, rapports de grâce",Autre fonction pendant la Commune,Profession militaire,Fonction pendant la Commune,Marques,Communard particulièrement recherché par les autorités (motif),Précédemment inculpé à la suite des événements de Juin 1848 accès à la fiche du site dédié
0,https://communards-1871.fr/index.php?page=fich...,Aab – Pierre Eugène Informations personnelles ...,"Paris, Seine",25/04/1835,36 ans,36 ans,célibataire ou en concubinage - 1 enfant(s) dé...,Jacques,Chocat Joséphine,Paris - Seine 38 Rue Montreuil (de) (quartie...,...,8J 12,COL H 69,BB/24/734,C//3103,NaN,NaN,NaN,NaN,NaN,NaN
1,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Adr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Paris - Seine,...,8J 226,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Lembeye, Pyrénées (Basses)",11/10/1835,36 ans,36 ans,marié(e),Dominique,Lavallière Bernarde,Paris - Seine 17 Route Asnières (d’) (quarti...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Tarbes, Pyrénées (Hautes)",03/10/1854,18 ans,17 ans,célibataire ou en concubinage,Pierre,Catherine,Paris - Seine 64 Avenue Clichy (de) (quartie...,...,8J 311,COL H 653,BB/24/834,NaN,Garde national au 91e puis au 207e bataillon.,NaN,NaN,NaN,NaN,NaN
4,https://communards-1871.fr/index.php?page=fich...,Abadie – Ismaël Isaac Informations personnelle...,"Vierzon, Cher",20/06/1820,NaN,51 ans,NaN,NaN,NaN,Paris - Seine 17 Rue Moreau (quartier Quinze...,...,8J 132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
# format of Garde nationale -> {some rank} - bataillon n {number}

# format of Garde nationale -> {some rank} - bataillon n {number}

col = 'Garde nationale'
s = df[col]

# rank = text before the first " -"
df['national_guard_rank'] = s.where(s.isna(), s.str.split(' -').str[0].str.strip())

# battalion = last number (everything after the last space -> extract digits at end)
df['national_guard_batallion'] = pd.to_numeric(s.where(s.isna(), s.str.extract(r'(\d+)\s*$')[0]), errors='coerce')

df.head()

,Profile_URL,Raw_Bio_Text,Lieu de naissance,Date de naissance déclarée,Âge déclaré,Âge calculé au 31 mai 1871,État civil,Père,Mère,Adresse,...,"Archives nationales, dossiers de grâce","Archives nationales, rapports de grâce",Autre fonction pendant la Commune,Profession militaire,Fonction pendant la Commune,Marques,Communard particulièrement recherché par les autorités (motif),Précédemment inculpé à la suite des événements de Juin 1848 accès à la fiche du site dédié,national_guard_rank,national_guard_batallion
0,https://communards-1871.fr/index.php?page=fich...,Aab – Pierre Eugène Informations personnelles ...,"Paris, Seine",25/04/1835,36 ans,36 ans,célibataire ou en concubinage - 1 enfant(s) dé...,Jacques,Chocat Joséphine,Paris - Seine 38 Rue Montreuil (de) (quartie...,...,BB/24/734,C//3103,NaN,NaN,NaN,NaN,NaN,NaN,Capitaine,206.0
1,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Adr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Paris - Seine,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Adjudant major,57.0
2,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Lembeye, Pyrénées (Basses)",11/10/1835,36 ans,36 ans,marié(e),Dominique,Lavallière Bernarde,Paris - Seine 17 Route Asnières (d’) (quarti...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Garde,132.0
3,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Tarbes, Pyrénées (Hautes)",03/10/1854,18 ans,17 ans,célibataire ou en concubinage,Pierre,Catherine,Paris - Seine 64 Avenue Clichy (de) (quartie...,...,BB/24/834,NaN,Garde national au 91e puis au 207e bataillon.,NaN,NaN,NaN,NaN,NaN,Garde,207.0
4,https://communards-1871.fr/index.php?page=fich...,Abadie – Ismaël Isaac Informations personnelle...,"Vierzon, Cher",20/06/1820,NaN,51 ans,NaN,NaN,NaN,Paris - Seine 17 Rue Moreau (quartier Quinze...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [77]:
import re

df = df.loc[:, ~df.columns.duplicated()].copy()

def parse_paris_address(addr):
    result = {
        "add_city": None,
        "add_department": None,
        "add_house_number": None,
        "add_street_raw": None,
        "add_street": None,
        "add_quartier": None,
    }

    if pd.isna(addr):
        return pd.Series(result)

    addr = re.sub(r"\s+", " ", str(addr)).strip()

    # City + department
    # normalize various apostrophes and non-breaking spaces so "d’" becomes "d'"
    addr = addr.replace("\u00A0", " ").replace("\u2019", "'").replace("\u2018", "'").replace("’", "'")

    m = re.match(
        r"^\s*(?P<city>[A-Za-zÀ-ÿ'\- ]+?)\s*-\s*(?P<department>[A-Za-zÀ-ÿ'\- ]+?)\s+(?P<rest>.+)$",
        addr,
    )
    if m:
        result["add_city"] = m.group("city").strip()
        result["add_department"] = m.group("department").strip()
        rest = m.group("rest").strip()
    else:
        rest = addr

    # Quartier
    m = re.search(r"\(quartier\s+([^)]+)\)", rest, flags=re.I)
    if m:
        result["add_quartier"] = m.group(1).strip()
        rest = re.sub(r"\(quartier\s+[^)]+\)", "", rest, flags=re.I).strip()
        pattern = r"^(?P<t>(?:Rue|Route|Boulevard|Avenue|Faubourg|Place|Impasse|Passage|Chemin|Quai|Square|Cité|Cour|Ruelle))\s+(?P<n>.+?)\s*\(\s*(?P<a>de|du|des|d['’])\s*\)\s*$"

        def _fix_article(m):
            t = m.group("t")
            name = m.group("n").strip()
            art = m.group("a").replace("'", "’")
            if art.lower().startswith("d’"):
                return f"{t} {art}{name}"
            else:
                return f"{t} {art} {name}"

        rest = re.sub(pattern, _fix_article, rest, flags=re.I)
    # House number
    m = re.match(r"^(?P<num>\d+\s*(?:bis|ter|quater|er)?)\s+(?P<street>.+)$", rest, flags=re.I)
    if m:
        result["add_house_number"] = re.sub(r"\s+", "", m.group("num"))
        rest = m.group("street").strip()

    result["add_street_raw"] = rest

    # Normalize street name
    street_types = r"(Rue|Route|Boulevard|Avenue|Faubourg|Place|Impasse|Passage|Chemin|Quai|Square|Cité|Cour|Ruelle)"
    m = re.match(rf"^{street_types}\s+(.+?)\s+\((de|du|des|d['’])\)$", rest, flags=re.I)

    if m:
        street_type = m.group(1).strip()
        street_name = m.group(2).strip()
        article = m.group(3).strip().replace("'", "’")
        if article.lower().startswith("d'") or article.lower().startswith("d’"):
            result["add_street"] = f"{street_type} {article}{street_name}"
        else:
            result["add_street"] = f"{street_type} {article} {street_name}"
    else:
        normalized = re.sub(r"[()]", "", rest)
        normalized = re.sub(r"\s+", " ", normalized).strip()
        result["add_street"] = normalized

    return pd.Series(result)

parsed = df["Adresse"].apply(parse_paris_address)

for c in parsed.columns:
    df[c] = parsed[c]

print(df.head())


                                         Profile_URL  \
0  https://communards-1871.fr/index.php?page=fich...   
1  https://communards-1871.fr/index.php?page=fich...   
2  https://communards-1871.fr/index.php?page=fich...   
3  https://communards-1871.fr/index.php?page=fich...   
4  https://communards-1871.fr/index.php?page=fich...   

                                        Raw_Bio_Text  \
0  Aab – Pierre Eugène Informations personnelles ...   
1  Abadie – Antoine Informations personnelles Adr...   
2  Abadie – Antoine Informations personnelles Lie...   
3  Abadie – Antoine Informations personnelles Lie...   
4  Abadie – Ismaël Isaac Informations personnelle...   

            Lieu de naissance Date de naissance déclarée Âge déclaré  \
0                Paris, Seine                 25/04/1835      36 ans   
1                         NaN                        NaN         NaN   
2  Lembeye, Pyrénées (Basses)                 11/10/1835      36 ans   
3   Tarbes, Pyrénées (Hautes)         

In [78]:
# format -> {num} ans
df['age'] = pd.to_numeric(df['Âge calculé au 31 mai 1871'].str.extract(r'(\d+)')[0], errors='coerce')

In [79]:
percent_paris = df['Adresse'].astype(str).str.contains(r'\bParis\b', na=False).mean() * 100
print(f"{percent_paris:.2f}%")

92.94%


In [80]:
df.head()

,Profile_URL,Raw_Bio_Text,Lieu de naissance,Date de naissance déclarée,Âge déclaré,Âge calculé au 31 mai 1871,État civil,Père,Mère,Adresse,...,Précédemment inculpé à la suite des événements de Juin 1848 accès à la fiche du site dédié,national_guard_rank,national_guard_batallion,add_city,add_department,add_house_number,add_street_raw,add_street,add_quartier,age
0,https://communards-1871.fr/index.php?page=fich...,Aab – Pierre Eugène Informations personnelles ...,"Paris, Seine",25/04/1835,36 ans,36 ans,célibataire ou en concubinage - 1 enfant(s) dé...,Jacques,Chocat Joséphine,Paris - Seine 38 Rue Montreuil (de) (quartie...,...,NaN,Capitaine,206.0,Paris,Seine,38,Rue Montreuil (de),Rue de Montreuil,Sainte Marguerite,36.0
1,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Adr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Paris - Seine,...,NaN,Adjudant major,57.0,NaN,NaN,NaN,Paris - Seine,Paris - Seine,NaN,NaN
2,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Lembeye, Pyrénées (Basses)",11/10/1835,36 ans,36 ans,marié(e),Dominique,Lavallière Bernarde,Paris - Seine 17 Route Asnières (d’) (quarti...,...,NaN,Garde,132.0,Paris,Seine,17,Route Asnières (d'),Route d’Asnières,Plaine de Monceaux,36.0
3,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Tarbes, Pyrénées (Hautes)",03/10/1854,18 ans,17 ans,célibataire ou en concubinage,Pierre,Catherine,Paris - Seine 64 Avenue Clichy (de) (quartie...,...,NaN,Garde,207.0,Paris,Seine,64,Avenue Clichy (de),Avenue de Clichy,Epinettes,17.0
4,https://communards-1871.fr/index.php?page=fich...,Abadie – Ismaël Isaac Informations personnelle...,"Vierzon, Cher",20/06/1820,NaN,51 ans,NaN,NaN,NaN,Paris - Seine 17 Rue Moreau (quartier Quinze...,...,NaN,NaN,NaN,Paris,Seine,17,Rue Moreau,Rue Moreau,Quinze Vingts,51.0


In [81]:
# df.to_csv('data/communards_clean.csv', index=False)

In [82]:
print(df.columns)

Index(['Profile_URL', 'Raw_Bio_Text', 'Lieu de naissance',
       'Date de naissance déclarée', 'Âge déclaré',
       'Âge calculé au 31 mai 1871', 'État civil', 'Père', 'Mère', 'Adresse',
       'Profession civile', 'Type d’activité', 'Branche', 'Garde nationale',
       'Date d’arrestation', 'Archives du Service historique de la Défense',
       'Archives nationales d’Outre-mer',
       'Archives nationales, dossiers de grâce',
       'Archives nationales, rapports de grâce',
       'Autre fonction pendant la Commune', 'Profession militaire',
       'Fonction pendant la Commune', 'Marques',
       'Communard particulièrement recherché par les autorités (motif)',
       'Précédemment inculpé à la suite des événements de Juin 1848 accès à la fiche du site dédié',
       'national_guard_rank', 'national_guard_batallion', 'add_city',
       'add_department', 'add_house_number', 'add_street_raw', 'add_street',
       'add_quartier', 'age'],
      dtype='str')


In [1]:
import pandas as pd 
df = pd.read_csv('data/geocoded_1871_addresses_checkpoint.csv')
df.head()

,_checkpoint_id,Profile_URL,Raw_Bio_Text,Lieu de naissance,Date de naissance déclarée,Âge déclaré,Âge calculé au 31 mai 1871,État civil,Père,Mère,...,add_house_number,add_street_raw,add_street,add_quartier,age,arrondissement,latitude,longitude,confidence_score,match_type
0,0,https://communards-1871.fr/index.php?page=fich...,Aab – Pierre Eugène Informations personnelles ...,"Paris, Seine",25/04/1835,36 ans,36 ans,célibataire ou en concubinage - 1 enfant(s) dé...,Jacques,Chocat Joséphine,...,38,Rue Montreuil (de),Rue deMontreuil,Sainte Marguerite,36.0,11.0,48.889437,2.333749,0.481985,housenumber
1,1,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Adr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,Paris - Seine,Paris - Seine,NaN,NaN,NaN,48.854446,2.336780,0.373809,street
2,2,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Lembeye, Pyrénées (Basses)",11/10/1835,36 ans,36 ans,marié(e),Dominique,Lavallière Bernarde,...,17,Route Asnières (d’),Route d’Asnières,Plaine de Monceaux,36.0,17.0,47.868098,-0.268180,0.514968,housenumber
3,3,https://communards-1871.fr/index.php?page=fich...,Abadie – Antoine Informations personnelles Lie...,"Tarbes, Pyrénées (Hautes)",03/10/1854,18 ans,17 ans,célibataire ou en concubinage,Pierre,Catherine,...,64,Avenue Clichy (de),Avenue deClichy,Epinettes,17.0,17.0,48.827525,2.379933,0.636295,housenumber
4,4,https://communards-1871.fr/index.php?page=fich...,Abadie – Ismaël Isaac Informations personnelle...,"Vierzon, Cher",20/06/1820,NaN,51 ans,NaN,NaN,NaN,...,17,Rue Moreau,Rue Moreau,Quinze Vingts,51.0,12.0,48.850171,2.372782,0.652525,housenumber


In [3]:
import pandas as pd
import folium
from IPython.display import display

plot_df = df.loc[df["arrondissement"].notna() & df["latitude"].notna() & df["longitude"].notna(), ["arrondissement", "latitude", "longitude"]].copy()
plot_df["latitude"] = pd.to_numeric(plot_df["latitude"], errors="coerce")
plot_df["longitude"] = pd.to_numeric(plot_df["longitude"], errors="coerce")
plot_df = plot_df.dropna(subset=["latitude", "longitude"])

if "Raw_Bio_Text" in df.columns:
    plot_df["person_name"] = df.loc[plot_df.index, "Raw_Bio_Text"].astype(str).str.extract(r"^(.*?)\s+Informations personnelles")[0].fillna("")
else:
    plot_df["person_name"] = ""

m = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=11.3,
    tiles="CartoDB positron",
    control_scale=False,
    zoom_control=False,
    scrollWheelZoom=False,
    dragging=True,
    prefer_canvas=True,
)

for _, row in plot_df.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,
        color="#111827",
        weight=0,
        fill=True,
        fill_color="#111827",
        fill_opacity=0.18,
        tooltip=row["person_name"] or None,
    ).add_to(m)

m

: 

: 